In [ ]:
# Base imports
import os
import pickle
import re

# Compute imports
import numpy as np
import pandas as pd
import scipy
from tqdm.notebook import tqdm, trange
from collections import defaultdict

# Plotting imports
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
from plotly import express as px
import matplotlib.patches as mpatches

# ML import
from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error, median_absolute_error
from sklearn.metrics.pairwise import cosine_similarity

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['svg.fonttype'] = 'none'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rcParams['font.family'] = 'sans-serif'
sns.set_style('ticks')
matplotlib.rcParams['text.color'] = '#000000'
matplotlib.rcParams['axes.labelcolor'] = '#000000'
matplotlib.rcParams['xtick.color'] = '#000000'
matplotlib.rcParams['ytick.color'] = '#000000'

In [ ]:
DF_GENES = '../../data/processed/panaroo_output/gene_presence_absence.Rtab'
ENRICHED_METADATA = '../../data/metadata/enriched_metadata.csv'
DF_EGGNOG = '../../data/processed/df_eggnog.csv'

DF_CORE_COMPLETE = '../../data/processed/CAR_genomes/df_core_panaroo.pickle'
DF_ACC_COMPLETE = '../../data/processed/CAR_genomes/df_acc_panaroo.pickle'
DF_RARE_COMPLETE = '../../data/processed/CAR_genomes/df_rare_panaroo.pickle'

L_BINARIZED = '../../data/processed/nmf-outputs/L_binarized.csv'
A_BINARIZED = '../../data/processed/nmf-outputs/A_binarized.csv'
L_MATRIX = '../../data/processed/nmf-outputs/L.csv'
A_MATRIX = '../../data/processed/nmf-outputs/A.csv'

In [ ]:
gene_locs_acc = pd.read_csv('acc_gene_location.csv', index_col=0)
gene_locs = pd.read_csv('complete_gene_location.csv', index_col=0)

In [ ]:
df_rare = pd.read_pickle(DF_RARE_COMPLETE)
df_acc = pd.read_pickle(DF_ACC_COMPLETE)
df_core = pd.read_pickle(DF_CORE_COMPLETE)

In [ ]:
metadata = pd.read_csv(ENRICHED_METADATA, index_col=0, dtype='object')

display( metadata.shape, metadata.head())

In [ ]:
# Load in (full) P matrix
df_genes = pd.read_csv(DF_GENES, sep='\t', index_col='Gene')

# Filter metadata for Complete sequences only
metadata_complete = metadata[metadata.genome_status == 'Complete'] # filter for only Complete sequences

# Filter P matrix for Complete sequences only
df_genes_complete = df_genes[metadata_complete.genome_id].copy()
df_genes_complete.fillna(0, inplace=True) # replace N/A with 0
df_genes_complete = df_genes_complete.astype('int8') # densify & typecast to int8 for space and compute reasons
inCompleteseqs = df_genes_complete.sum(axis=1) > 0 # filter for genes found in complete sequences
df_genes_complete = df_genes_complete[inCompleteseqs]

df_genes_complete.shape

In [ ]:
# Load in eggNOG annotations
df_eggnog = pd.read_csv(DF_EGGNOG, index_col=0)
df_eggnog.fillna('-', inplace=True)

display(
    df_eggnog.shape,
    df_eggnog.head()
)

In [ ]:
# Load in A_binarized matrix
A_binarized = pd.read_csv(A_BINARIZED, index_col=0)
A_binarized

In [ ]:
# Load in L_binarized matrix
L_binarized = pd.read_csv(L_BINARIZED, index_col=0)
L_binarized

In [ ]:
phylon_order = [
    'mobile-1',
    'mobile-4',
    'mobile-2',
    'mobile-3',
    'mobile-10',
    'mobile-7',
    'mobile-6',
    'mobile-5',
    'mobile-8',
    'mobile-9',
    'roggenkampii',
    'asburiae-1',
    'asburiae-2',
    'cancerogenous',
    'kobei',
    'bugandensis',
    'mori',
    'ludwigii',
    'cloacae',
    'hormaechei-steigerwaltii-2',
    'hormaechei-steigerwaltii-4',
    'hormaechei-steigerwaltii-1',
    'hormaechei-steigerwaltii-3',
    'hormaechei-hoffmannii-1',
    'hormaechei-hoffmannii-2',
    'hormaechei-hoffmannii-3',
    'hormaechei-hormaechei',
    'hormaechei-oharae',
    'hormaechei-xiangfangensis-2',
    'hormaechei-xiangfangensis-1',
    'hormaechei-xiangfangensis-3',
]

characterized_order = [x for x in phylon_order if 'mobile' not in x]

# Useful functions

In [ ]:
def unique_genes_by_phylon(df: pd.DataFrame) -> dict:
    '''
    This function identifies unique genes for each phylon in a L_binarized.

    Parameters:
    df (pd.DataFrame, L_binarized): A dataframe where columns are phylon names, 
                       row indices are gene names, 
                       and values are 1 or 0 indicating the presence of the gene in the phylon.

    Returns:
    dict: A dictionary where keys are phylon names and values are lists of genes 
          that are unique to each phylon.
    '''
    unique_genes = {}

    # Iterate through each phylon (column)
    for phylon in df.columns:
        # Get genes present in the current phylon
        genes_in_phylon = df.index[df[phylon] == 1].tolist()

        # Find unique genes by ensuring they are not present in any other phylon
        unique_genes[phylon] = [gene for gene in genes_in_phylon if df.loc[gene].sum() == 1]

    return unique_genes


def generate_dendrogram_and_split(X: pd.DataFrame, linkage_method = "ward", linkage_metric='euclidean') -> tuple:
    '''
    This function generates a linkage matrix based on hierarchical clustering and returns the membership
    of each cluster in terms of indices corresponding the the columns of the input matrix. It also returns
    a tree of the cluster values which represents the structure of the dendrogram.

    Parameters:
        X (pd.DataFrame): A dataframe reprsenting the data to be clustered (ex. the L_binarized  matrix)
        linkage_method (str): a linkage method accepted as input to scipy's linkage function
        linkage_metric (str): a distance metric to be used for linkage calculations by scipy
    
    Returns:
        clusters (dict): a dictionary containing lists of the clusters present in the generated 
            linkage matrix with each cluster represented by an intiger. The first n intigers 
            correspond to the n columns of X (assuming X is m x n)
        split_tree (dict): a dictionary of dictionaries representing the tree structure of the above clusters
        links (numpy array): a numpy array of the linkage matrix output by the hierarchical clustering
    '''
    
    X = X.T
    links = scipy.cluster.hierarchy.linkage(X, method=linkage_method, metric = linkage_metric)
    
    
    clusters = {x:[x] for x in range(len(X.index))}
    
    for i, (left, right, _, _) in enumerate(links):
        clusters[len(clusters)] = clusters[left] + clusters[right]
    
    
    def track_split(cluster, clusters, links):
        if len(clusters[cluster]) == 1:
            return cluster
    
        row = cluster - len(links) - 1
        left_child = int(links[row][0])
        right_child = int(links[row][1])
        
        return {cluster:{left_child:track_split(left_child, clusters, links), right_child:track_split(right_child, clusters, links)}}
    
    
    split_tree = track_split(max(clusters.keys()), clusters, links)

    return clusters, split_tree, links

def get_gene_sets(X: pd.DataFrame, clusters: dict, splits: dict, cluster: str|int) -> tuple:
    """
        This function takes in a matrix X (m by n) of genes by phylons (or strains), a dendrogram structure (as output by the
        `generate_dendrogram_and_split` function, the clusters output by this same function, and a cluster of interest from 
        clusters. It returns four lists boolean values relevant to this cluster's gene content for each category (described below).

        Parameters:
            X (pd.DataFrame): A dataframe reprsenting the data to be clustered (ex. the L_binarized  matrix)
            clusters (dict): a dictionary containing lists of the clusters present in the generated 
                linkage matrix with each cluster represented by an intiger. The first n intigers 
                correspond to the n columns of X (assuming X is m x n)
            split_tree (dict): a dictionary of dictionaries representing the tree structure of the above clusters
            cluster (int or str): cluster of interest from  clusters

        Returns:
            ubiquitous_exclusive_genes (list): a boolean list of length m representing the genes in the index of X which 
                are found in all members of this cluster and not in any other leaf nodes of the tree
            exclusive_genes (list): a boolean list of length m representing the genes in the index of X which 
                are found in any of the  members of this cluster and not in any other leaf nodes of the tree
            total_split_genes (list): a boolean list of length m representing the genes in the index of X which 
                are found in any of the  members of this cluster
            total_ubiquitous_genes (list): a boolean list of length m representing the genes in the index of X which 
                are found in all of the  members of this cluster
    """
    cluster_members = clusters[cluster]
    cluster_size = len(cluster_members)

    df_array = X.values
    member_cols = np.array(cluster_members)
    other_cols = np.array([x for x in range(X.shape[1]) if x not in member_cols])

    if other_cols.size == 0:  # Handle edge case
        ubiquitous_exclusive_genes = (df_array[:, member_cols] == 1).all(axis=1)
        exclusive_genes = (df_array[:, member_cols] == 1).any(axis=1)
        total_split_genes = (df_array[:, member_cols] == 1).any(axis=1)
        total_ubiquitous_genes = (df_array[:, member_cols] == 1).all(axis=1)
        return ubiquitous_exclusive_genes, exclusive_genes, total_split_genes, total_ubiquitous_genes
    
    ubiquitous_exclusive_genes = (df_array[:, member_cols] == 1).all(axis=1) & (df_array[:, other_cols] == 0).all(axis=1)
    exclusive_genes = (df_array[:, member_cols] == 1).any(axis=1) & (df_array[:, other_cols] == 0).all(axis=1)
    total_split_genes = (df_array[:, member_cols] == 1).any(axis=1)
    total_ubiquitous_genes = (df_array[:, member_cols] == 1).all(axis=1)
    
    return ubiquitous_exclusive_genes, exclusive_genes, total_split_genes, total_ubiquitous_genes

def generate_split_genes(X : pd.DataFrame, linkage_method : str = "ward", linkage_metric : str= 'euclidean'):
    """
        Function to calculate the  number of the categories of genes described in `get_gene_sets` for a hierarchical
        clustering of the provided X (m x n) matrix. Returns a dataframe containing cluster association with each
        of these gene set sizes and the clusters and split tree from the generated clustering.

        Parameters:
            X (pd.DataFrame): A dataframe reprsenting the data to be clustered (ex. the L_binarized  matrix)
            linkage_method (str): a linkage method accepted as input to scipy's linkage function
            linkage_metric (str): a distance metric to be used for linkage calculations by scipy

        Returns:
            df_split_values (dataframe): dataframe with the clusters as indices containing the number of genes
                in this cluster for each of the 4 gene sets as described by `get_gene_sets`
            clusters (dict): a dictionary containing lists of the clusters present in the generated 
                linkage matrix with each cluster represented by an intiger. The first n intigers 
                correspond to the n columns of X (assuming X is m x n)
            split_tree (dict): a dictionary of dictionaries representing the tree structure of the above clusters
            links (numpy array): a numpy array of the linkage matrix output by the hierarchical clustering
    """
    clusters, split_tree, links = generate_dendrogram_and_split(X, linkage_method = linkage_method, linkage_metric= linkage_metric)
    
    df_split_values = pd.DataFrame(index = clusters.keys(), columns = ['ubiquitous_exclusive_genes', 'exclusive_genes', 
                            'total_split_genes','total_ubiquitous_genes'])
    
    for cluster in (list(clusters.keys())):
        df_split_values.loc[cluster] = [sum(x) for x in get_gene_sets(X, clusters, split_tree, cluster)]

    return df_split_values, clusters, split_tree, links


def generate_split_genes_lists(X : pd.DataFrame, linkage_method : str = "ward", linkage_metric : str= 'euclidean'):
    """
        Function to calculate the gene sets of the categories of genes described in `get_gene_sets` for a hierarchical
        clustering of the provided X (m x n) matrix. Returns a dictionary containing a dictionary of the gene split
        lists for each split in the tree structure. 

        Parameters:
            X (pd.DataFrame): A dataframe reprsenting the data to be clustered (ex. the L_binarized  matrix)
            linkage_method (str): a linkage method accepted as input to scipy's linkage function
            linkage_metric (str): a distance metric to be used for linkage calculations by scipy

        Returns:
            split_genes_dict (dict): dictionary with the lists of genes for each category of genes for each split
            clusters (dict): a dictionary containing lists of the clusters present in the generated 
                linkage matrix with each cluster represented by an intiger. The first n intigers 
                correspond to the n columns of X (assuming X is m x n)
            split_tree (dict): a dictionary of dictionaries representing the tree structure of the above clusters
            links (numpy array): a numpy array of the linkage matrix output by the hierarchical clustering
    """

    clusters, split_tree, links = generate_dendrogram_and_split(X, linkage_method = linkage_method, linkage_metric= linkage_metric)
    
    split_gene_sets = defaultdict(dict)
    
    for cluster in (list(clusters.keys())):
        gene_sets = [X.index[x] for x in get_gene_sets(X, clusters, split_tree, cluster)]
        split_gene_sets[cluster]['ubiquitous_exclusive_genes'] = gene_sets[0]
        split_gene_sets[cluster]['exclusive_genes'] = gene_sets[1]
        split_gene_sets[cluster]['total_split_genes'] = gene_sets[2]
        split_gene_sets[cluster]['total_ubiquitous_genes'] = gene_sets[3]
    
    return split_gene_sets, clusters, split_tree, links

In [ ]:
def generate_phylon_dendrogram(L_binarized : pd.DataFrame, linkage_method : str = "ward", linkage_metric : str= 'euclidean',
                               labels : list|str = ['exclusive_genes'], ax : matplotlib.axes._axes.Axes= None,
                              orientation : str = 'left', text_offset: int = 10, label_color: str = 'red',
                              text_size : int = 7):
    """
        Generate a denddrogram with labels of the gene content at each of the non-leaf splits.

        Parameters:
            L_binarized (pd.DataFrame): gene x phylon matrix to cluster and make dendrogram
            linkage_method (str): a linkage method accepted as input to scipy's linkage function
            linkage_metric (str): a distance metric to be used for linkage calculations by scipy
            labels (list): list of labels from `df_split_values` including 'ubiquitous_exclusive_genes', 'exclusive_genes', 
                'total_split_genes','total_ubiquitous_genes', default is 'exclusive_genes'
            ax (matplotlib ax): ax to draw plot onto (optional)
            orientation (str): orientation of the dendrogram
            text_offset (int): offset of text (positive is right/down depending on orientataion)
            label_color (str): color for split labels
            text_size (int): font size for the text

        returns:
            ax (matplotlib ax): ax of the plot generated by the function
            df_stats (dataframe): dataframe with the clusters as indices containing the number of genes
                in this cluster for each of the 4 gene sets as described by `get_gene_sets` and the phylon content
                of each split (or leaf) in the tree
            split_genes_dict_labeled (dict): dictionary with the lists of genes for each category of genes for
                each split labeled with the associated split or phylon
    """
    if type(labels) == str:
        labels = [labels]

    if not ax:
        fig, ax = plt.subplots(figsize=(10, 6))

    assert orientation in ['left', 'right', 'top', 'bottom'], "Must use a valid orientation"
    valid_labels = ['ubiquitous_exclusive_genes', 'exclusive_genes', 'total_split_genes','total_ubiquitous_genes']
    assert all([x in valid_labels  for x in labels]), "Please use a list of labels from " + ', '.join(valid_labels) 

    df_split_values, clusters, split_tree, links = generate_split_genes(L_binarized, linkage_method = linkage_method,
                                                                       linkage_metric = linkage_metric)

    ddata = scipy.cluster.hierarchy.dendrogram(links, labels=L_binarized.columns, orientation=orientation, ax=ax)

    Z_distances = links[:, 2]  # The merge distances from linkage matrix

    split_labels = {}
    for icoord, dcoord in zip(ddata['icoord'], ddata['dcoord']):
        merge_height = dcoord[1]
    
        # Match the current drawing (merge_height) to a row in the linkage matrix
        z_idx = None
        for i, dist in enumerate(Z_distances):
            if np.isclose(dist, merge_height):
                z_idx = i
    

        label_idx = z_idx + L_binarized.shape[1]
        
        split_label_value = len(Z_distances) - z_idx
        split_labels[label_idx] = split_label_value
        
        text = "Split " + str(split_label_value) + ':\n'
        for label in labels:
            text = text + label + ': ' + str(df_split_values.loc[label_idx, label]) + '\n'
        text = text[:-1]
    
        x_pos = dcoord[1] - text_offset
        y_pos = (icoord[1] + icoord[2]) / 2

        vert_align = 'center'
        horz_align = 'left'
        if orientation == 'top' or orientation ==  'bottom':
            temp = x_pos
            x_pos = y_pos
            y_pos = temp
            vert_align = 'top'
            horz_align = 'center'
        
        ax.text(x_pos, y_pos, str(text), va=vert_align, ha=horz_align, fontsize=text_size, color=label_color)


    # Annotate leaves using tick positions and label order
    leaf_order = ddata['leaves']  # list of indices into original L_binarized.columns
    leaf_labels = ddata['ivl']    # list of leaf names in order of appearance

    if orientation in ['left', 'right']:
        tick_positions = ax.get_yticks()
    else:
        tick_positions = ax.get_xticks()

    for i, leaf_idx in enumerate(leaf_order):
        if orientation == 'left':
            x = 0 + text_offset
            y = tick_positions[i]
            va, ha = 'bottom', 'right'
        elif orientation == 'right':
            x = 0 - text_offset
            y = tick_positions[i]
            va, ha = 'bottom', 'left'
        elif orientation == 'top':
            x = tick_positions[i]
            y = 0 - text_offset
            va, ha = 'bottom', 'center'
        else:  # bottom
            x = tick_positions[i]
            y = 0 + text_offset
            va, ha = 'top', 'center'

        # Build the label text from df_split_values
        leaf_label_text = ''
        for label in labels:
            if 'ubiquitous' not in label:
                leaf_label_text += f"{label}: {df_split_values.loc[leaf_idx, label]}\n"
        leaf_label_text = leaf_label_text.strip()

        ax.text(x, y, leaf_label_text, va=va, ha=ha, fontsize=text_size, color=label_color)

    # create output statistics
    df_stats = df_split_values.copy()

    split_gene_sets, _, _, _ = generate_split_genes_lists(L_binarized, linkage_method = linkage_method, linkage_metric = linkage_metric)

    split_gene_sets_labeled = {}
    cluster_membership = pd.Series()
    for ind in df_stats.index:
        cluster_membership[ind] = ';'.join(list(L_binarized.columns[clusters[ind]]))
    df_stats['split_membership'] = cluster_membership
    
    new_inds = []
    df_stats = df_stats.loc[sorted(df_stats.index)[::-1]]
    for ind in df_stats.index:
        if ind in split_labels.keys():
            new_inds.append("Split " + str(split_labels[ind]))
            split_gene_sets_labeled["Split " + str(split_labels[ind])] = split_gene_sets[ind]
        else:
            new_inds.append(L_binarized.columns[ind])
            split_gene_sets_labeled[L_binarized.columns[ind]] = split_gene_sets[ind]
    df_stats.index = new_inds
    
    return ax, df_stats, split_gene_sets_labeled

# Phylon Dendrogram and Splits

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(15,15))
ax, df_stats, split_gene_sets_labeled = generate_phylon_dendrogram(L_binarized, text_offset=0, 
                                                    labels = ['exclusive_genes'], orientation = 'left', ax = ax)

# Gene Sets for Each Phylon

In [ ]:
for phylon in phylon_order:
    phylon_genes = df_eggnog.loc[split_gene_sets_labeled[phylon]['total_split_genes']]
    phylon_genes['phylon_exclusive'] = df_eggnog.apply(lambda x: x.name in split_gene_sets_labeled[phylon]['exclusive_genes'], axis = 1)
    phylon_genes.to_csv('../../data/phylon_gene_lists/' + phylon + '.csv')

In [ ]:
acc_info = pd.merge(df_eggnog.loc[df_acc.index], L_binarized[phylon_order], left_index=True, right_index=True)
acc_info['phylon_list'] = ""

for gene in acc_info.index:
    phylon_str = []
    for phylon in phylon_order:
        if L_binarized.loc[gene, phylon]:
            phylon_str.append(phylon)
    phylon_str = ';'.join(phylon_str)
    acc_info.loc[gene, 'phylon_list'] = phylon_str

In [ ]:
strain_info = pd.merge(metadata_complete.set_index('genome_id'), A_binarized.T[phylon_order], left_index=True, right_index=True)
strain_info['phylon_list'] = ""

for strain in strain_info.index:
    phylon_str = []
    for phylon in phylon_order:
        if A_binarized.T.loc[strain, phylon]:
            phylon_str.append(phylon)
    phylon_str = ';'.join(phylon_str)
    strain_info.loc[strain, 'phylon_list'] = phylon_str

In [ ]:
strain_info.to_csv('../../data/phylon_gene_lists/strain_info.csv')
acc_info.to_csv('../../data/phylon_gene_lists/gene_info.csv')